### Paths

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

observations_path = Path("/scratch/s1803229/ATL08_parquets/S01.parquet")
experiment = Path("/scratch/s1803229/fsm2_ensembles/S01/nov25")
lookup_path = Path("../../output/classification_uncertainty_EgEv.nc")
output = experiment / "comparison.parquet"

probs = ["prob_no_snow", "prob_ground_snow", "prob_ground_canopy_snow"]

### Creating grouped parquet

In [20]:
MIN_EG_RATIO = 1.15
MIN_DATA_QUANTITY = 15

grouped_path = observations_path.with_stem(observations_path.stem + "_grouped")

if grouped_path.exists():
    obs = pd.read_parquet(grouped_path)
else:
    keys = ["site_id", "resource", "date", "lat", "lon"]
    stats = {
        "pvpg": "mean", "pv": "max", "pg": "max", "data_quantity": "max",
        "snr": "mean", "layer_flag": "mean", "msw_flag": "mean",
        "pv_ratio_mean": "mean", "pv_ratio_max": "mean",
    }
    columns = keys + list(stats) + ["beam_str", "outlier", "Eg", "Ev"]
    raw = pd.read_parquet(observations_path, columns=columns)

    aggs = {name: (name, method) for name, method in stats.items()}
    for strength in ("strong", "weak"):
        keep = raw["beam_str"].eq(strength) & raw["outlier"].eq(1)
        for variable in ("Eg", "Ev"):
            name = f"{variable}_{strength}"
            raw[name] = raw[variable].where(keep)
            aggs[name] = (name, "median")

    obs = raw.groupby(keys, sort=False, observed=True).agg(
        **aggs, segment_rows=("Eg", "size")
    ).reset_index()
    del raw

    obs["date"] = pd.to_datetime(obs["date"], format="%d/%m/%Y")
    temporary = grouped_path.with_suffix(".partial.parquet")
    obs.to_parquet(temporary, index=False, compression="zstd")
    temporary.replace(grouped_path)

n_before = len(obs)
eg_ratio = obs["Eg_strong"].div(obs["Eg_weak"].where(obs["Eg_weak"] > 0))
keep = (
    np.isfinite(eg_ratio)
    & eg_ratio.gt(MIN_EG_RATIO)
    & obs["data_quantity"].ge(MIN_DATA_QUANTITY)
)
obs = obs.loc[keep].copy()

print(f"Kept {len(obs):,} of {n_before:,} groups ({n_before - len(obs):,} excluded)")
obs.head()

Kept 168,133 of 267,970 groups (99,837 excluded)


,site_id,resource,date,lat,lon,pvpg,pv,pg,data_quantity,snr,layer_flag,msw_flag,pv_ratio_mean,pv_ratio_max,Eg_strong,Ev_strong,Eg_weak,Ev_weak,segment_rows
4,S01,ATL03_20181014221052_02490102_007_01.h5,2018-10-14,52.438739,-103.883591,0.559572,2.530629,4.522436,28.0,3.430134,0.0,0.0,2.604134,2.604134,2.609756,1.011905,1.000000,0.358571,52
5,S01,ATL03_20181014221052_02490102_007_01.h5,2018-10-14,52.447748,-103.884907,0.882614,3.446965,3.905405,21.0,4.065967,0.0,0.0,2.699189,2.699189,2.926829,0.829268,1.130435,0.269231,40
7,S01,ATL03_20181014221052_02490102_007_01.h5,2018-10-14,52.483784,-103.891228,0.065943,1.051754,15.949372,19.0,4.584270,0.0,0.0,1.973991,1.973991,3.690476,0.785714,1.093750,0.515152,32
9,S01,ATL03_20181014221052_02490102_007_01.h5,2018-10-14,52.501802,-103.893977,0.090836,1.057613,11.643051,20.0,5.376593,0.0,0.0,2.264174,2.264174,3.738095,0.702381,1.300000,0.266667,27
10,S01,ATL03_20181014221052_02490102_007_01.h5,2018-10-14,52.510811,-103.895334,0.192312,1.320060,6.864161,20.0,5.101227,0.0,0.0,2.781543,2.781543,3.328904,0.710801,1.258065,0.212121,29


### Assign probabilities and nearest model hour

In [21]:
if "observation_time" in obs:
    obs["observation_time"] = pd.to_datetime(obs["observation_time"], utc=True)
else:
    stamp = obs["resource"].str.extract(r"ATL0[38]_(\d{14})", expand=False)
    obs["observation_time"] = pd.to_datetime(stamp, format="%Y%m%d%H%M%S", utc=True)

if obs["observation_time"].isna().any():
    raise ValueError("Missing observation timestamp")

obs["model_time"] = obs["observation_time"].dt.round("h").dt.tz_localize(None)

query = {
    name: xr.DataArray(obs[name].to_numpy(), dims="observation")
    for name in ("Eg_strong", "Ev_strong")
}
with xr.open_dataset(lookup_path) as lookup:
    for name in probs:
        obs[name] = lookup[name].interp(query).values

valid = np.isfinite(obs[probs]).all(axis=1)
obs["icesat_class"] = pd.array(
    np.where(valid, obs[probs].to_numpy().argmax(axis=1), np.nan), dtype="Int8"
)

### Browse FSM2 models with ICESat-2 overlay

In [22]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from IPython.display import clear_output, display
from matplotlib.colors import BoundaryNorm, ListedColormap, to_rgb
from matplotlib.lines import Line2D

# Run after the notebook cell that sets observation_time, model_time and probs.
site = experiment.parent.name
members = (experiment / "members.txt").read_text().split()
files = {member: experiment / member / f"{site}.nc" for member in members}

class_names = ["No snow", "Ground snow", "Ground + canopy snow"]
colors = ["#6b6b6b", "#2878b5", "#e68120"]
rgb = np.array([to_rgb(color) for color in colors])
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

tracks = obs[["observation_time", "model_time", "lat", "lon", *probs]].dropna(
    subset=["observation_time", "model_time", "lat", "lon"]
).copy()
if tracks.empty:
    raise ValueError("No dated ICESat-2 observations with coordinates.")

# Keep the actual overpass day even if rounding moves the model hour past midnight.
tracks["day"] = pd.to_datetime(tracks["observation_time"], utc=True).dt.date
by_slot = tracks.groupby(["day", "model_time"], sort=True)
slots = sorted(by_slot.groups)

options = []
for index, (overpass_day, model_time) in enumerate(slots):
    label = f"{model_time:%Y-%m-%d %H:%M} UTC"
    if overpass_day != model_time.date():
        label += f" (overpass {overpass_day})"
    options.append((label, index))

when = widgets.Dropdown(
    options=options,
    description="Model hour",
    layout=widgets.Layout(width="420px"),
)
previous = widgets.Button(description="←", tooltip="Previous overpass hour",
                          layout=widgets.Layout(width="42px"))
following = widgets.Button(description="→", tooltip="Next overpass hour",
                           layout=widgets.Layout(width="42px"))
out = widgets.Output()

handles = [
    Line2D([], [], linestyle="none", marker="o", color=color, label=name)
    for color, name in zip(colors, class_names)
]
handles += [
    Line2D([], [], linestyle="none", marker="o", markerfacecolor="white",
           markeredgecolor="#333333", label="High uncertainty"),
    Line2D([], [], linestyle="none", marker="x", color="black", label="Unclassified"),
]

def draw(change=None):
    index = when.value
    previous.disabled = index == 0
    following.disabled = index == len(slots) - 1

    overpass_day, t = slots[index]
    points = by_slot.get_group((overpass_day, t))

    # Fade each classified ICESat-2 point toward white as uncertainty increases.
    p = points[probs].to_numpy(dtype=float)
    valid = np.isfinite(p).all(axis=1) & (p.sum(axis=1) > 0)
    p = p[valid]
    p = p / p.sum(axis=1, keepdims=True)
    fade = np.clip((1 - p.max(axis=1)) / (2 / 3), 0, 1)
    point_rgb = rgb[p.argmax(axis=1)] * (1 - fade[:, None]) + fade[:, None]
    order = np.argsort(fade)
    classified = points.loc[valid].iloc[order]
    unknown = points.loc[~valid]

    with out:
        clear_output(wait=True)
        nrows = (len(files) + 1) // 2
        fig, axes = plt.subplots(nrows, 2, figsize=(12, 4.5 * nrows), squeeze=False)

        for ax, (member, path) in zip(axes.flat, files.items()):
            ax.set_title(member)
            if not path.is_file():
                ax.text(.5, .5, "Assembled NetCDF missing", ha="center",
                        transform=ax.transAxes)
                ax.set_axis_off()
                continue

            with xr.open_dataset(path) as ds:
                if t not in ds.indexes["time"]:
                    ax.text(.5, .5, "Hour outside model output", ha="center",
                            transform=ax.transAxes)
                    ax.set_axis_off()
                    continue
                frame = ds[["ground_snow_cover_fraction", "canopy_snow_mass",
                            "ground_snow_mass"]].sel(time=t).load()

            ground = frame.ground_snow_cover_fraction
            canopy = frame.canopy_snow_mass
            finite = np.isfinite(ground) & np.isfinite(canopy) & np.isfinite(frame.ground_snow_mass)
            snow = xr.where(ground >= 0.5, xr.where(canopy > 0.5, 2, 1), 0).where(finite)

            snow.plot.pcolormesh(
                ax=ax, x="longitude", y="latitude", cmap=cmap, norm=norm,
                alpha=.45, add_colorbar=False,
            )
            ax.set_autoscale_on(False)
            ax.scatter(classified.lon, classified.lat, c=point_rgb[order], s=16,
                       edgecolors="#222222", linewidths=.3, zorder=3)
            ax.scatter(unknown.lon, unknown.lat, c="black", marker="x",
                       s=10, linewidths=.6, zorder=4)
            ax.set_title(member)
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.set_aspect(1 / np.cos(np.deg2rad(float(frame.latitude.mean()))))

        for ax in list(axes.flat)[len(files):]:
            ax.set_visible(False)

        fig.suptitle(f"{site}  |  FSM2 {t:%Y-%m-%d %H:%M} UTC  |  "
                     f"{len(points):,} ICESat-2 observations")
        fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False)
        fig.tight_layout(rect=(0, .05, 1, .96))
        plt.show()
        plt.close(fig)

def go_previous(_):
    if when.index > 0:
        when.index -= 1

def go_next(_):
    if when.index < len(slots) - 1:
        when.index += 1

previous.on_click(go_previous)
following.on_click(go_next)
when.observe(draw, names="value")

display(widgets.VBox([widgets.HBox([previous, when, following]), out]))
draw()

# needs more editing below this I think to work the way I want

### Compare all members and write parquet (long processing, just run once)
ground snow fraction has to be above 50% for binary to be 1, canopy snow has to be above 0.5 kg per meter squared for binary to be 1.

In [7]:
from contextlib import ExitStack
from tempfile import TemporaryDirectory
import pyarrow as pa
import pyarrow.parquet as pq

members = (experiment / "members.txt").read_text().split()
site = experiment.parent.name
variables = ["ground_snow_cover_fraction", "ground_snow_mass", "canopy_snow_mass"]
cell_hour = ["model_latitude", "model_longitude", "model_time"]

with TemporaryDirectory(dir=experiment) as tmp:
    pending = Path(tmp) / "comparison.parquet"

    with ExitStack() as stack:
        for i, member in enumerate(members):
            path = experiment / member / f"{site}.nc"
            rows = obs.copy()

            with xr.open_dataset(path) as model:
                indices = {}
                inside = np.ones(len(rows), dtype=bool)

                for dim, col in [("latitude", "lat"), ("longitude", "lon")]:
                    centres = model[dim].values
                    spacing = np.abs(np.diff(centres))
                    if not len(spacing) or not np.allclose(spacing, spacing[0]):
                        raise ValueError(f"Expected a regular {dim} grid")

                    idx = model.indexes[dim].get_indexer(rows[col], method="nearest")
                    rows[f"model_{dim}"] = centres[idx]
                    inside &= np.abs(rows[col] - centres[idx]) <= spacing[0] / 2 + 1e-8
                    indices[dim] = idx

                time_idx = model.indexes["time"].get_indexer(rows["model_time"])
                inside &= time_idx >= 0
                indices["time"] = np.maximum(time_idx, 0)

                selected = model[variables].isel({
                    dim: xr.DataArray(idx, dims="observation")
                    for dim, idx in indices.items()
                }).load()
                for name in variables:
                    rows[name] = np.where(inside, selected[name].values, np.nan)

            ground = rows["ground_snow_cover_fraction"] >= 0.5
            canopy = rows["canopy_snow_mass"] > 0.5
            valid_model = np.isfinite(rows[variables]).all(axis=1)
            classes = np.where(ground, np.where(canopy, 2, 1), 0)
            rows["model_class"] = pd.array(
                np.where(valid_model, classes, np.nan), dtype="Int8"
            )

            matched = valid_model & rows["icesat_class"].notna()
            rows["matched"] = matched
            rows["comparison_weight"] = 0.0
            count = rows.loc[matched].groupby(["resource", *cell_hour])["resource"].transform("size")
            rows.loc[matched, "comparison_weight"] = 1 / count
            rows["member"] = member
            rows["model_file"] = str(path)
            rows["ground_threshold_fraction"] = 0.5
            rows["canopy_threshold_kg_m2"] = 0.5

            table = pa.Table.from_pandas(rows, preserve_index=False)
            if i == 0:
                writer = stack.enter_context(pq.ParquetWriter(pending, table.schema))
            writer.write_table(table)
            print(f"{member}: {matched.sum():,} matched groups")

    pending.replace(output)

clm5_cap1_loss1: 261,396 matched groups
hp98_pine_cap1_loss1: 261,396 matched groups
hp98_spruce_cap1_loss1: 261,396 matched groups
noahmp_cap1_loss1: 261,396 matched groups


### Load comparison and rank members

In [8]:
comparison = pd.read_parquet(output)
comparison["scheme"] = comparison["member"].str.split("_cap", n=1).str[0]

matched = comparison.loc[comparison["matched"]].copy()
matched["weighted_correct"] = (
    matched["comparison_weight"] * matched["model_class"].eq(matched["icesat_class"])
)

scores = matched.groupby(["scheme", "member"]).agg(
    groups=("matched", "size"),
    correct=("weighted_correct", "sum"),
    weight=("comparison_weight", "sum"),
)
scores["agreement_pct"] = 100 * scores["correct"] / scores["weight"]
scores[["groups", "agreement_pct"]].sort_values("agreement_pct", ascending=False).round(2)

,,groups,agreement_pct
scheme,member,,
clm5,clm5_cap1_loss1,261396,83.58
noahmp,noahmp_cap1_loss1,261396,83.06
hp98_spruce,hp98_spruce_cap1_loss1,261396,78.55
hp98_pine,hp98_pine_cap1_loss1,261396,78.39


### Confusion Matrix

In [9]:
member = "clm5_cap1_loss1"
one = matched.loc[matched["member"].eq(member)]

matrix = pd.crosstab(
    one["icesat_class"], one["model_class"],
    values=one["comparison_weight"], aggfunc="sum",
).reindex(index=[0, 1, 2], columns=[0, 1, 2]).fillna(0)

matrix.div(matrix.sum(axis=1), axis=0).mul(100).round(1)

model_class,0,1,2
icesat_class,,,
0,95.0,4.9,0.1
1,16.2,79.8,4.0
2,4.9,84.3,10.8


### Inspect 1km cells sharing same cell-hour

In [10]:
cell_hour = ["model_latitude", "model_longitude", "model_time"]
groups = one.groupby(cell_hour)
counts = groups.size().sort_values(ascending=False)
display(counts.head())

if len(counts):
    display(groups.get_group(counts.index[0])[
        ["resource", "lat", "lon", *probs, "icesat_class", "model_class",
         "ground_snow_cover_fraction", "canopy_snow_mass", "comparison_weight"]
    ])

model_latitude  model_longitude  model_time         
53.2            -103.0           2024-09-28 14:00:00    27
                -104.7           2022-07-12 05:00:00    26
54.2            -103.6           2022-08-24 15:00:00    25
                -104.8           2019-04-18 13:00:00    25
                -104.9           2024-12-25 23:00:00    25
dtype: int64

,resource,lat,lon,prob_no_snow,prob_ground_snow,prob_ground_canopy_snow,icesat_class,model_class,ground_snow_cover_fraction,canopy_snow_mass,comparison_weight
216483,ATL03_20240928141316_01882502_007_01.h5,53.204505,-102.950693,0.989099,0.010887,0.000014,0,0,0.0,0.0,0.037037
216484,ATL03_20240928141316_01882502_007_01.h5,53.213514,-102.952027,0.987739,0.012247,0.000014,0,0,0.0,0.0,0.037037
216485,ATL03_20240928141316_01882502_007_01.h5,53.222523,-102.953461,0.975452,0.024495,0.000054,0,0,0.0,0.0,0.037037
216486,ATL03_20240928141316_01882502_007_01.h5,53.231532,-102.954918,0.981616,0.018343,0.000041,0,0,0.0,0.0,0.037037
216487,ATL03_20240928141316_01882502_007_01.h5,53.240541,-102.956367,0.969904,0.030036,0.000060,0,0,0.0,0.0,0.037037
216488,ATL03_20240928141316_01882502_007_01.h5,53.249550,-102.957820,0.975503,0.024451,0.000047,0,0,0.0,0.0,0.037037
216760,ATL03_20240928141316_01882502_007_01.h5,53.150450,-102.989887,0.984692,0.015250,0.000058,0,0,0.0,0.0,0.037037
216761,ATL03_20240928141316_01882502_007_01.h5,53.159459,-102.990959,0.975491,0.024384,0.000125,0,0,0.0,0.0,0.037037
216762,ATL03_20240928141316_01882502_007_01.h5,53.168468,-102.992803,0.990730,0.009258,0.000012,0,0,0.0,0.0,0.037037
216763,ATL03_20240928141316_01882502_007_01.h5,53.177477,-102.994169,0.987386,0.012602,0.000012,0,0,0.0,0.0,0.037037
